# 1. ANN 구조 파악

이 노트북은 인공신경망(ANN, Artificial Neural Network)의 기본 개념을 이해하고, MNIST 손글씨 숫자 데이터셋을 이용해 실제로 분류 모델을 학습해보는 예제를 담고 있습니다.

> GPU가 잡히지 않아 `Using device: cpu`가 나온다면 먼저 `00_Torch_GPU_셋업.ipynb`를 확인하세요.

## 목차

- [1-1. ANN에 대한 기본 설명](#1-1.-ANN에-대한-기본-설명)
- [1-2. MNIST 숫자 데이터셋 ANN 학습 예시](#1-2.-MNIST-숫자-데이터셋-ANN-학습-예시)


## 1-1. ANN에 대한 기본 설명

ANN(Artificial Neural Network)은 인간의 신경망 구조에서 아이디어를 얻은 머신러닝 모델입니다. 입력 데이터를 여러 층(layer)을 거치게 하면서, 점점 더 유용한 특징을 추출하고 최종적으로 원하는 출력을 예측합니다.

### 1) 기본 구성

- **입력층(Input Layer)**: 데이터를 처음 받아들이는 층입니다.
- **은닉층(Hidden Layer)**: 입력값을 가공하고 의미 있는 표현으로 바꾸는 층입니다.
- **출력층(Output Layer)**: 최종 예측 결과를 내는 층입니다.

### 2) 뉴런의 동작 방식

하나의 뉴런은 입력값들에 각각 가중치(weight)를 곱하고, 편향(bias)을 더한 뒤, 활성화 함수(activation function)를 통과시켜 출력값을 만듭니다.

수식으로 쓰면 다음과 같습니다.

$$
z = w_1x_1 + w_2x_2 + ... + w_nx_n + b
$$

$$
a = f(z)
$$

여기서:

- $x$: 입력값
- $w$: 가중치
- $b$: 편향
- $f$: 활성화 함수
- $a$: 뉴런의 출력값

### 3) 왜 은닉층이 필요한가?

은닉층이 없다면 모델은 단순한 선형 관계만 학습하기 쉽습니다. 하지만 실제 이미지, 음성, 텍스트 데이터는 훨씬 복잡하므로, 비선형성을 제공하는 은닉층과 활성화 함수가 중요합니다.

### 4) 대표적인 활성화 함수

- **ReLU**: 가장 많이 쓰이는 활성화 함수 중 하나입니다.
- **Sigmoid**: 출력이 0과 1 사이로 제한됩니다.
- **Softmax**: 여러 클래스 중 하나를 선택하는 다중 분류에서 자주 사용됩니다.

### 5) 학습 과정

ANN은 보통 다음 순서로 학습됩니다.

1. 입력 데이터를 모델에 넣어 예측값을 계산합니다. (순전파, Forward Propagation)
2. 예측값과 정답의 차이를 손실 함수(Loss Function)로 계산합니다.
3. 오차를 뒤로 전달하며 각 가중치가 얼마나 수정되어야 하는지 계산합니다. (역전파, Backpropagation)
4. 옵티마이저(Optimizer)가 가중치를 업데이트합니다.

### 6) MNIST에서 ANN이 하는 일

MNIST 데이터는 28x28 크기의 손글씨 숫자 이미지입니다. 이를 펼치면 784개의 숫자 입력이 됩니다. ANN은 이 784차원 입력을 받아서 최종적으로 0부터 9까지 어떤 숫자인지 분류합니다.


## 1-2. MNIST 숫자 데이터셋 ANN 학습 예시

이번 예제에서는 PyTorch를 사용해서 간단한 ANN 분류기를 만들고 MNIST를 학습합니다.

### 실습 목표

- MNIST 데이터를 불러온다.
- ANN 모델을 정의한다.
- 학습과 평가를 수행한다.
- 예측 결과를 직접 확인한다.


In [ ]:
# 필요한 라이브러리가 없다면 아래 주석을 해제해서 설치하세요.
# !pip install torch torchvision matplotlib


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

print('PyTorch version:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


### 1) 데이터 준비

MNIST 이미지는 28x28 크기의 흑백 이미지입니다. ANN에 넣기 전에 텐서로 바꾸고, 일반적으로 학습 안정성을 위해 정규화(normalization)를 적용합니다.


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print('Train samples:', len(train_dataset))
print('Test samples:', len(test_dataset))


In [ ]:
images, labels = next(iter(train_loader))
print('배치 이미지 shape:', images.shape)
print('배치 라벨 shape:', labels.shape)

plt.figure(figsize=(10, 4))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i].squeeze(), cmap='gray')
    plt.title(f'label: {labels[i].item()}')
    plt.axis('off')
plt.tight_layout()
plt.show()


### 2) ANN 모델 정의

이미지 한 장은 `(1, 28, 28)` 형태지만, 완전연결층(Linear Layer)에 넣기 위해 `784 = 28 x 28` 형태로 펼쳐서 사용합니다.

여기서는 다음과 같은 구조를 사용합니다.

- 입력층: 784
- 은닉층 1: 256
- 은닉층 2: 128
- 출력층: 10


In [ ]:
class ANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.model(x)

model = ANN().to(device)
print(model)


### 3) 손실 함수와 옵티마이저 설정

다중 분류 문제이므로 손실 함수는 `CrossEntropyLoss`를 사용합니다. 옵티마이저는 기본적으로 많이 사용하는 `Adam`을 사용합니다.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


### 4) 학습 함수와 평가 함수 정의

학습 단계에서는 모델이 배치 단위로 예측을 수행하고, 손실을 계산한 뒤 역전파를 통해 가중치를 업데이트합니다.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


### 5) 모델 학습

처음에는 5 epoch 정도만 실행해도 ANN이 MNIST를 꽤 잘 분류하는 것을 확인할 수 있습니다.


In [ ]:
epochs = 5
history = {
    'train_loss': [],
    'train_acc': [],
    'test_loss': [],
    'test_acc': []
}

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    print(
        f'Epoch [{epoch + 1}/{epochs}] | '
        f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | '
        f'Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}'
    )


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['test_loss'], label='Test Loss')
plt.title('Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['test_acc'], label='Test Acc')
plt.title('Accuracy Curve')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()


### 6) 테스트 데이터로 예측 확인

학습이 끝난 뒤에는 실제 테스트 이미지 몇 장을 꺼내서 모델이 어떤 숫자로 예측하는지 직접 볼 수 있습니다.


In [ ]:
model.eval()
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    outputs = model(images)
    predictions = outputs.argmax(dim=1)

images = images.cpu()
labels = labels.cpu()
predictions = predictions.cpu()

plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(images[i].squeeze(), cmap='gray')
    plt.title(f'True: {labels[i].item()} / Pred: {predictions[i].item()}')
    plt.axis('off')

plt.tight_layout()
plt.show()


## 정리

이번 노트북에서 확인한 핵심은 다음과 같습니다.

- ANN은 입력층, 은닉층, 출력층으로 구성됩니다.
- MNIST 이미지는 28x28이지만, ANN에 넣을 때는 784차원 벡터로 펼쳐서 사용할 수 있습니다.
- 은닉층과 ReLU 활성화 함수를 통해 비선형 패턴을 학습할 수 있습니다.
- `CrossEntropyLoss`와 `Adam` 조합으로 손쉽게 분류 모델을 학습할 수 있습니다.

다음 단계에서는 CNN과 비교하면서, 이미지 데이터에서 왜 CNN이 ANN보다 더 강력한지 이어서 학습해볼 수 있습니다.
